# Material Selection Protocol — 5-Step Method
**Plastic Design Calculators — Notebook 6**

---

## Part 1: Theory and Governing Equations

### 1.1 The Polymer Selection Challenge

With thousands of polymer grades available, material selection without a structured filter quickly becomes subjective and error-prone. The **5-Step Method** transforms qualitative application demands into quantifiable property thresholds, which are then evaluated deterministically against a material database.

### 1.2 The 5-Step Sequence

| Step | Action |
|---|---|
| **1** | List all product functional demands |
| **2** | Translate demands for Component A into measurable engineering thresholds |
| **3** | Translate demands for Component B into measurable engineering thresholds |
| **4** | Filter material database → candidates for Component A |
| **5** | Filter material database → candidates for Component B |

### 1.3 Property Translation Matrix

The translation matrix maps functional needs to testable properties:

| Functional Requirement | Engineering Property | Direction |
|---|---|---|
| High flexibility | Flexural modulus | $< E_{\mathrm{threshold}}$ |
| High stiffness | Tensile modulus | $> E_{\mathrm{threshold}}$ |
| Heat resistance | Heat Deflection Temperature (HDT) | $> T_{\mathrm{threshold}}$ |
| Impact resistance | Notched Izod impact | $> I_{\mathrm{threshold}}$ |
| Elongation / ductility | Elongation at yield | $> \epsilon_{\mathrm{threshold}}$ |
| Regulatory compliance | FDA / food-contact approval | $= $ True |

### 1.4 Boolean Masking

The `pandas` engine executes Boolean AND of all active constraints:

$$\text{mask} = \bigcap_i \left(\text{property}_i \; \{<, >, =\} \; \text{threshold}_i\right)$$

Only materials satisfying **all** constraints simultaneously are retained.

### Assumptions
- Property values are mean values at 23 °C, dry-as-moulded
- FDA approval is a binary flag (not a specific regulation number)
- No chemical resistance or UL94 flammability filter implemented

---
## Part 2: Variable Definitions and Unit Handling

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import sympy as sp
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from utils.unit_registry import ureg, Q_
from utils.material_db import MATERIAL_DB

# ── Step 1: Product demands ────────────────────────────────────────────────────
PRODUCT_DEMANDS = [
    "FDA food-contact approval required",
    "Component A (flexible bowl): must flex repeatedly without fracture",
    "Component A: must withstand boiling water contact (100°C continuous)",
    "Component B (rigid handle): must resist bending under 50 N grip load",
    "Component B: dimensional stability at 150°C (dishwasher)",
]

print("Product Demands:")
for i, d in enumerate(PRODUCT_DEMANDS, 1):
    print(f"  {i}. {d}")

# ── Build material DataFrame ───────────────────────────────────────────────────
df_all = pd.DataFrame(MATERIAL_DB)
print(f"\nMaterial database loaded: {len(df_all)} entries")
df_all.head()

---
## Part 3: Computation Engine

In [ ]:
# ── Step 2: Translation matrix — Component A (flexible bowl) ──────────────────
constraints_A = [
    {'property': 'flexural_modulus_MPa',   'operator': '<',  'threshold': 1500,  'unit': 'MPa',
     'requirement': 'High Flexibility'},
    {'property': 'elongation_yield_pct',   'operator': '>',  'threshold': 15.0,  'unit': '%',
     'requirement': 'Ductility / cold-draw capability'},
    {'property': 'HDT_C',                  'operator': '>',  'threshold': 100.0, 'unit': '°C',
     'requirement': 'Boiling water resistance'},
    {'property': 'FDA_approved',           'operator': '==', 'threshold': True,  'unit': '—',
     'requirement': 'FDA food-contact compliance'},
]

# ── Step 3: Translation matrix — Component B (rigid handle) ───────────────────
constraints_B = [
    {'property': 'tensile_modulus_MPa',    'operator': '>',  'threshold': 5000,  'unit': 'MPa',
     'requirement': 'High Stiffness'},
    {'property': 'HDT_C',                  'operator': '>',  'threshold': 150.0, 'unit': '°C',
     'requirement': 'Dishwasher / heat resistance'},
    {'property': 'notched_izod_kJ_m2',     'operator': '>',  'threshold': 50.0,  'unit': 'kJ/m²',
     'requirement': 'Impact / drop resistance'},
    {'property': 'FDA_approved',           'operator': '==', 'threshold': True,  'unit': '—',
     'requirement': 'FDA food-contact compliance'},
]

print("Constraints defined for both components.")

In [ ]:
def apply_constraints(df, constraints, **kwargs):
    """Filter a material DataFrame by Boolean AND of all constraints.

    Args:
        df (pandas.DataFrame): Full material database.
        constraints (list[dict]): Each dict has 'property', 'operator', 'threshold'.
        **kwargs: Reserved for weighted scoring extensions (Pugh matrix).

    Returns:
        pandas.DataFrame: Filtered subset of df satisfying all constraints.
    """
    mask = pd.Series([True] * len(df), index=df.index)
    for c in constraints:
        col = c['property']
        op  = c['operator']
        thr = c['threshold']
        if op == '<':
            mask &= df[col] < thr
        elif op == '<=':
            mask &= df[col] <= thr
        elif op == '>':
            mask &= df[col] > thr
        elif op == '>=':
            mask &= df[col] >= thr
        elif op == '==':
            mask &= df[col] == thr
    return df[mask].copy()


def build_translation_table(constraints, component_label, **kwargs):
    """Render the property translation matrix as a styled DataFrame.

    Args:
        constraints (list[dict]): Constraint dictionaries.
        component_label (str): Label for display.
        **kwargs: Reserved for LaTeX export mode.

    Returns:
        pandas.DataFrame: Human-readable translation matrix.
    """
    rows = []
    for c in constraints:
        rows.append({
            'Component': component_label,
            'Functional Requirement': c['requirement'],
            'Engineering Property': c['property'].replace('_', ' ').title(),
            'Operator': c['operator'],
            'Threshold': f"{c['threshold']} {c['unit']}",
        })
    return pd.DataFrame(rows)


print("Filter functions defined.")

In [ ]:
# ── Steps 4 & 5: Execute filtering ────────────────────────────────────────────
df_A = apply_constraints(df_all, constraints_A)
df_B = apply_constraints(df_all, constraints_B)

table_A = build_translation_table(constraints_A, 'Component A (Flexible Bowl)')
table_B = build_translation_table(constraints_B, 'Component B (Rigid Handle)')

print("Property Translation Matrix — Component A:")
print(table_A.to_string(index=False))
print()
print("Property Translation Matrix — Component B:")
print(table_B.to_string(index=False))

display_cols = ['name', 'flexural_modulus_MPa', 'tensile_modulus_MPa',
                'HDT_C', 'notched_izod_kJ_m2', 'elongation_yield_pct', 'FDA_approved']

print(f"\n{'='*60}")
print(f"  COMPONENT A CANDIDATES ({len(df_A)} found):")
print(f"{'='*60}")
print(df_A[display_cols].to_string(index=False) if len(df_A) > 0 else "  No materials satisfy all constraints.")

print(f"\n{'='*60}")
print(f"  COMPONENT B CANDIDATES ({len(df_B)} found):")
print(f"{'='*60}")
print(df_B[display_cols].to_string(index=False) if len(df_B) > 0 else "  No materials satisfy all constraints.")

---
## Part 4: Data Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# ── Plot 1: Flexural Modulus vs HDT for all materials (Component A context) ───
ax1 = axes[0]
all_names = df_all['name'].tolist()
cand_A    = set(df_A['name'].tolist())

for _, row in df_all.iterrows():
    is_cand = row['name'] in cand_A
    ax1.scatter(row['HDT_C'], row['flexural_modulus_MPa'],
                color='limegreen' if is_cand else 'lightgrey',
                edgecolors='black', s=80, zorder=3)
    ax1.annotate(row['name'], (row['HDT_C'], row['flexural_modulus_MPa']),
                 fontsize=6, ha='left', va='bottom',
                 color='darkgreen' if is_cand else 'grey')

# threshold lines
ax1.axhline(1500, color='steelblue', ls='--', lw=1.5, label='Flex mod limit 1500 MPa')
ax1.axvline(100,  color='tomato',    ls='--', lw=1.5, label='HDT limit 100°C')
ax1.set_xlabel('HDT [°C]')
ax1.set_ylabel('Flexural Modulus [MPa]')
ax1.set_title('Component A — Flexibility vs Heat Resistance\n(green = candidate)')
ax1.legend(fontsize=7)
ax1.grid(True, alpha=0.3)

# ── Plot 2: Tensile Modulus vs Impact Strength (Component B context) ──────────
ax2 = axes[1]
cand_B = set(df_B['name'].tolist())

for _, row in df_all.iterrows():
    is_cand = row['name'] in cand_B
    ax2.scatter(row['tensile_modulus_MPa'], row['notched_izod_kJ_m2'],
                color='steelblue' if is_cand else 'lightgrey',
                edgecolors='black', s=80, zorder=3)
    ax2.annotate(row['name'], (row['tensile_modulus_MPa'], row['notched_izod_kJ_m2']),
                 fontsize=6, ha='left', va='bottom',
                 color='steelblue' if is_cand else 'grey')

ax2.axvline(5000, color='limegreen', ls='--', lw=1.5, label='Tensile mod limit 5000 MPa')
ax2.axhline(50,   color='tomato',    ls='--', lw=1.5, label='Impact limit 50 kJ/m²')
ax2.set_xlabel('Tensile Modulus [MPa]')
ax2.set_ylabel('Notched Izod Impact [kJ/m²]')
ax2.set_title('Component B — Stiffness vs Impact\n(blue = candidate)')
ax2.legend(fontsize=7)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('06_material_selection_output.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Part 5: Design Rule Validation

In [ ]:
def badge(passed):
    return "\033[92m  PASS  \033[0m" if passed else "\033[91m  FAIL  \033[0m"

pass_A = len(df_A) > 0
pass_B = len(df_B) > 0

print("═" * 65)
print("  DESIGN RULE VALIDATION — MATERIAL SELECTION")
print("═" * 65)
print(f"\n  Component A — Viable candidates found: {badge(pass_A)}")
if pass_A:
    for _, row in df_A.iterrows():
        print(f"    ✓  {row['name']}")
        print(f"         Flex mod: {row['flexural_modulus_MPa']} MPa   "
              f"HDT: {row['HDT_C']}°C   "
              f"Elong: {row['elongation_yield_pct']}%   "
              f"FDA: {row['FDA_approved']}")
else:
    print("    ✗ No candidates — relax one or more constraints.")

print(f"\n  Component B — Viable candidates found: {badge(pass_B)}")
if pass_B:
    for _, row in df_B.iterrows():
        print(f"    ✓  {row['name']}")
        print(f"         Tensile mod: {row['tensile_modulus_MPa']} MPa   "
              f"HDT: {row['HDT_C']}°C   "
              f"Impact: {row['notched_izod_kJ_m2']} kJ/m²   "
              f"FDA: {row['FDA_approved']}")
else:
    print("    ✗ No candidates — consider glass-filled grades or relaxing HDT.")

print("═" * 65)
if pass_A and pass_B:
    print("  ✓ OVERALL: Both components have viable material candidates.")
else:
    print("  ✗ OVERALL: One or more components have no viable candidates.")
print("═" * 65)